<a href="https://colab.research.google.com/github/mohiuddin2806-lang/machine-learning-/blob/main/CN7031_week_2_u2975262.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title ▶ Set up Hadoop 3.5.0 and start one NameNode + one DataNode
# Import paths, operating-system utilities, download and checksum tools.
from pathlib import Path
import os
import subprocess
import urllib.request
import urllib.error
import hashlib
import re
import tarfile
import time

# Import XML tools to create Hadoop configuration files.
from xml.etree import ElementTree as ET

# Import the HTTP client used to check the NameNode and DataNode web services.
import requests

# Set a persistent path within the current temporary Colab runtime.
LAB = Path('/content/cn7031-week2-hdfs')
# Create the working directory when it does not exist.
LAB.mkdir(parents=True, exist_ok=True)
# Specify the Hadoop installation and Java paths.
HADOOP = LAB / 'hadoop-3.5.0'
JAVA = Path('/usr/lib/jvm/java-17-openjdk-amd64')

# Run a system command and stop with a useful error if it fails.
def checked(args, timeout=600):
    # Capture standard output and errors as readable text.
    p = subprocess.run(args, text=True, stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, timeout=timeout)
    # Check the exit status before using the command's result.
    if p.returncode:
        raise RuntimeError(p.stdout[-5000:])
    # Return the printed output to the caller.
    return p.stdout

# Install Java 17 only when it is not already available.
if not (JAVA / 'bin/java').exists():
    # Update package metadata for the Colab VM.
    print('Installing Java …')
    checked(['apt-get', 'update', '-qq'])
    # Install the headless JDK required by Hadoop.
    checked(['apt-get', 'install', '-y', '-qq', 'openjdk-17-jdk-headless'])

# Download and verify Hadoop only once per current VM.
if not (HADOOP / '.verified').exists():
    # Identify the archive destination and Apache download URL.
    print('Downloading Hadoop from Apache (about 554 MB) …')
    archive = LAB / 'hadoop-3.5.0.tar.gz'
    base = 'https://downloads.apache.org/hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz'
    # Keep track of progress so the display is not too noisy.
    last = [-1]

    # Report download progress in roughly 20% increments.
    def progress(count, block, total):
        # A known total size lets us calculate the percentage.
        if total > 0:
            percent = min(100, int(count * block * 100 / total))
            # Print only when a new progress interval is reached.
            if percent // 20 > last[0]:
                print(f'  Download: {percent}%')
                last[0] = percent // 20

    # Try the official mirror; use the Apache archive if unavailable.
    try:
        urllib.request.urlretrieve(base, archive, reporthook=progress)
    except (urllib.error.URLError, TimeoutError):
        # Use an archive URL instead of a missing current-release URL.
        base = base.replace('downloads.apache.org', 'archive.apache.org/dist')
        urllib.request.urlretrieve(base, archive, reporthook=progress)

    # Read Apache's published SHA-512 digest for this exact archive.
    with urllib.request.urlopen(base + '.sha512', timeout=60) as response:
        digest = re.findall(r'(?i)\b[0-9a-f]{128}\b', response.read().decode())
    # Calculate the SHA-512 digest of the downloaded file.
    with archive.open('rb') as stream:
        actual = hashlib.file_digest(stream, 'sha512').hexdigest()
    # Abort setup if the digests do not match.
    assert len(digest) == 1 and digest[0].lower() == actual, 'Download checksum failed. Stop here.'
    # Extract the verified archive safely.
    print('Checksum verified. Installing Hadoop …')
    with tarfile.open(archive) as bundle:
        bundle.extractall(LAB, filter='data')
    # Record successful verification and remove the compressed archive.
    (HADOOP / '.verified').write_text(actual)
    archive.unlink()

# Set Java and Hadoop environment variables for child processes.
os.environ.update(JAVA_HOME=str(JAVA), HADOOP_HOME=str(HADOOP),
                  HADOOP_CONF_DIR=str(HADOOP / 'etc/hadoop'),
                  HADOOP_LOG_DIR=str(LAB / 'logs'), HADOOP_PID_DIR=str(LAB / 'pids'))
# Put Hadoop and Java executables on the shell PATH.
os.environ['PATH'] = f'{HADOOP}/bin:{JAVA}/bin:' + os.environ['PATH']
# Make separate folders for namespace metadata, blocks, logs and process IDs.
for name in ('namenode', 'datanode', 'logs', 'pids'):
    (LAB / name).mkdir(exist_ok=True)

# Write a Hadoop XML configuration file from a property dictionary.
def config(name, values):
    # Create the configuration root element.
    root = ET.Element('configuration')
    # Add each property name and its value.
    for key, value in values.items():
        property_node = ET.SubElement(root, 'property')
        ET.SubElement(property_node, 'name').text = key
        ET.SubElement(property_node, 'value').text = str(value)
    # Save the resulting configuration into Hadoop's config directory.
    ET.ElementTree(root).write(HADOOP / 'etc/hadoop' / name, encoding='utf-8')

# Point HDFS clients at the local NameNode using port 9000.
config('core-site.xml', {
    'fs.defaultFS': 'hdfs://127.0.0.1:9000',
    'hadoop.tmp.dir': str(LAB / 'tmp')
})
# Assign local storage directories and a default replication factor of one.
config('hdfs-site.xml', {
    'dfs.namenode.name.dir': (LAB / 'namenode').as_uri(),
    'dfs.datanode.data.dir': (LAB / 'datanode').as_uri(),
    'dfs.replication': 1,
    # Bind RPC and health-report endpoints to the VM's loopback address.
    'dfs.namenode.rpc-address': '127.0.0.1:9000',
    'dfs.namenode.rpc-bind-host': '127.0.0.1',
    'dfs.namenode.http-address': '127.0.0.1:9870',
    'dfs.namenode.http-bind-host': '127.0.0.1',
    # Keep DataNode transfer, HTTP and IPC endpoints on localhost.
    'dfs.datanode.address': '127.0.0.1:9866',
    'dfs.datanode.http.address': '127.0.0.1:9864',
    'dfs.datanode.ipc.address': '127.0.0.1:9867',
    'dfs.datanode.hostname': '127.0.0.1',
    'dfs.client.use.datanode.hostname': 'true',
    'dfs.datanode.use.datanode.hostname': 'true',
})

# Read Hadoop's environment shell script before adding Java and heap settings.
env = HADOOP / 'etc/hadoop/hadoop-env.sh'
# Avoid appending duplicate configuration lines on reruns.
if 'export HADOOP_HEAPSIZE_MAX=512' not in env.read_text():
    env.write_text(env.read_text() +
                   f'\n# CN7031 HDFS environment\nexport JAVA_HOME={JAVA}\nexport HADOOP_HEAPSIZE_MAX=512\n')

# Format only when no NameNode filesystem already exists.
if not (LAB / 'namenode/current/VERSION').exists():
    # Never silently overwrite an unexpectedly populated storage directory.
    assert all(not any((LAB / name).iterdir()) for name in ('namenode', 'datanode')), \
        'Existing storage found. Do not reformat it.'
    # Give the new HDFS namespace a fixed educational cluster ID.
    checked(['hdfs', 'namenode', '-format', '-nonInteractive', '-clusterid', 'CID-cn7031-week2'])

# Start a service only if its local HTTP endpoint is not responding.
for daemon, port in [('namenode', 9870), ('datanode', 9864)]:
    try:
        # Raise an error on a missing or unhealthy service endpoint.
        requests.get(f'http://127.0.0.1:{port}/jmx', timeout=2).raise_for_status()
    except requests.RequestException:
        # Start the selected Hadoop daemon in the background.
        checked(['hdfs', '--daemon', 'start', daemon])

# Wait until the NameNode reports exactly one live DataNode.
for attempt in range(45):
    try:
        # Query NameNode's JMX metrics over its local HTTP endpoint.
        response = requests.get('http://127.0.0.1:9870/jmx',
                                params={'qry': 'Hadoop:service=NameNode,name=FSNamesystemState'},
                                timeout=3)
        response.raise_for_status()
        # Exit the retry loop when the teaching cluster is ready.
        if response.json()['beans'][0]['NumLiveDataNodes'] == 1:
            break
    except (requests.RequestException, KeyError, IndexError):
        # Retry while the services finish their startup sequence.
        pass
    # Pause two seconds before asking again.
    time.sleep(2)
else:
    # Stop if the DataNode never becomes available.
    raise RuntimeError('Startup failed. Inspect the Hadoop logs in /content/cn7031-week2-hdfs/logs.')

# Wait until the NameNode has left safe mode, allowing writes.
checked(['hdfs', 'dfsadmin', '-safemode', 'wait'], timeout=90)
# Confirm that HDFS is usable before the exercises begin.
print('\nHDFS is ready ✓')
print('Hadoop 3.5.0 · NameNode running · 1 DataNode running')


HDFS is ready ✓
Hadoop 3.5.0 · NameNode running · 1 DataNode running


In [ ]:
# List local Java processes to confirm that both HDFS services started.
!jps

40817 Jps
4821 NameNode
4822 DataNode


In [ ]:
# Import Path to locate the file uploaded to Colab.
from pathlib import Path
# Point to the uploaded file on Colab's LOCAL filesystem, not HDFS.
local_file = Path('/content/access_log_NASA')
# Stop with clear guidance if the upload has not been completed.
assert local_file.is_file(), 'Upload access_log_NASA into the /content folder in Colab first.'
# Display the filename and its exact byte size.
print(f'Local dataset: {local_file.name} ({local_file.stat().st_size:,} bytes)')

Local dataset: access_log_NASA (205,242,368 bytes)


In [ ]:
# Create both HDFS directories; -p allows them to exist already.
!hdfs dfs -mkdir -p /data /data_copy

In [ ]:
# Upload the LOCAL NASA log into HDFS; -f allows replacement on reruns.
!hdfs dfs -put -f /content/access_1og_NASA /data/

put: `/content/access_1og_NASA': No such file or directory


In [ ]:
# Upload the LOCAL NASA log into HDFS; -f allows replacement on reruns.
!hdfs dfs -put -f /content/access_log_NASA /data/

In [ ]:
# List HDFS directory contents and check the uploaded file size.
!hdfs dfs -ls /data

Found 2 items
-rw-r--r--   1 root supergroup  205242368 2026-10-20 04:37 /data/access_log_NASA
drwxr-xr-x   - root supergroup          0 2026-10-20 04:34 /data/content


In [ ]:
# Preview the start of the HDFS log without downloading the full file.
!hdfs dfs -head  /data/access_log_NASA

199.72.81.55 - - [01/Jul/1995:00:00:01 -0400] "GET /history/apollo/ HTTP/1.0" 200 6245
unicomp6.unicomp.net - - [01/Jul/1995:00:00:06 -0400] "GET /shuttle/countdown/ HTTP/1.0" 200 3985
199.120.110.21 - - [01/Jul/1995:00:00:09 -0400] "GET /shuttle/missions/sts-73/mission-sts-73.html HTTP/1.0" 200 4085
burger.letters.com - - [01/Jul/1995:00:00:11 -0400] "GET /shuttle/countdown/liftoff.html HTTP/1.0" 304 0
199.120.110.21 - - [01/Jul/1995:00:00:11 -0400] "GET /shuttle/missions/sts-73/sts-73-patch-small.gif HTTP/1.0" 200 4179
burger.letters.com - - [01/Jul/1995:00:00:12 -0400] "GET /images/NASA-logosmall.gif HTTP/1.0" 304 0
burger.letters.com - - [01/Jul/1995:00:00:12 -0400] "GET /shuttle/countdown/video/livevideo.gif HTTP/1.0" 200 0
205.212.115.106 - - [01/Jul/1995:00:00:12 -0400] "GET /shuttle/countdown/countdown.html HTTP/1.0" 200 3985
d104.aa.net - - [01/Jul/1995:00:00:13 -0400] "GET /shuttle/countdown/ HTTP/1.0" 200 3985
129.94.144.152 - - [01/Jul/1995:00:00:13 -0400] "GET / HTTP/1.0" 

In [ ]:
# Copy the HDFS file to a separate HDFS directory; -f overwrites an old archive.
!hdfs dfs -cp -f /data/access_log_NASA /data_copy/access_log_archive

In [ ]:
# Remove only the first HDFS path, after making the archive copy; -f ignores a missing path.
!hdfs dfs -rm -f /data/access_log_NASA

Deleted /data/access_log_NASA


In [ ]:
# Confirm the archive still exists in HDFS after the first path was deleted.
!hdfs dfs -ls  /data_copy

Found 1 items
-rw-r--r--   1 root supergroup  205242368 2026-10-20 04:38 /data_copy/access_log_archive


In [ ]:
# Download the surviving HDFS archive to a DIFFERENT local filename.
# -f allows rerunning the download without a destination-file conflict.
!hdfs dfs -get -f /data_copy/access_log_archive /content/access_log_from_hdfs

In [ ]:
#Show human-readable logical size and HDFS-accounted space for the archive.
!hdfs dfs -du -h /data_copy/access_log_archive

195.7 M  195.7 M  /data_copy/access_log_archive


In [ ]:
# Ask the NameNode for file health, block IDs and DataNode locations.
# -files lists files, -blocks shows HDFS blocks, and -locations shows their hosts.
!hdfs fsck /data_copy/access_log_archive -files -blocks -locations

Connecting to namenode via http://127.0.0.1:9870/fsck?ugi=root&files=1&blocks=1&locations=1&path=%2Fdata_copy%2Faccess_log_archive
FSCK started by root (auth:SIMPLE) from /127.0.0.1 for path /data_copy/access_log_archive at Tue Oct 20 04:47:39 UTC 2026

/data_copy/access_log_archive 205242368 bytes, replicated: replication=1, 2 block(s):  OK
0. BP-679200969-172.28.0.12-1792465818356:blk_1073749915_9091 len=134217728 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-1a6c30ba-d06c-4a1e-9f7d-3a43bb3bba31,DISK]]
1. BP-679200969-172.28.0.12-1792465818356:blk_1073749916_9092 len=71024640 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-1a6c30ba-d06c-4a1e-9f7d-3a43bb3bba31,DISK]]


Status: HEALTHY
 Number of data-nodes:	1
 Number of racks:		1
 Total dirs:			0
 Total symlinks:		0

Replicated Blocks:
 Total size:	205242368 B
 Total files:	1
 Total blocks (validated):	2 (avg. block size 102621184 B)
 Minimally replicated blocks:	2 (100.0 %)
 Over-replicated blocks:	0 (0.0 %)
 Under-

In [ ]:
#Inspect the NameNode's local metadata directory (not an HDFS pathname).
#-l shows details, -a includes hidden files, and -h formats sizes.
!ls -lah /content/cn7031-week2-hdfs/namenode/current


total 24K
drwx------ 2 root root 4.0K Oct 20 03:25 .
drwxr-xr-x 3 root root 4.0K Oct 20 03:25 ..
-rw-r--r-- 1 root root  399 Oct 20 03:25 fsimage_0000000000000000000
-rw-r--r-- 1 root root   62 Oct 20 03:25 fsimage_0000000000000000000.md5
-rw-r--r-- 1 root root    2 Oct 20 03:25 seen_txid
-rw-r--r-- 1 root root  191 Oct 20 03:25 VERSION


In [ ]:
# Import path to walk the DataNode's local block-storage directory.
from pathlib import Path
# Recursively list actual block-content files, excluding block checksum sidecars.
data_dir = Path('/content/cn7031-week2-hdfs/datanode')
block_files = sorted(p for p in data_dir.rglob('blk_*')
if p.is_file() and not p.name.endswith('.meta'))
#Display the number of physical block files stored on this DataNode.
print(f'DataNode block-content files:{len(block_files)}')
# Show a few sample paths and their physical byte sizes.
for path in block_files[:5]:
  print(f'{path.name}:{path.stat().st_size:,}bytes - {path}')

DataNode block-content files:0


In [ ]:
# Request the HOT storage policy on the surviving archived HDFS file.
!hdfs storagepolicies -setStoragePolicy -policy HOT -path /data_copy/access_log_archive

Set storage policy HOT on /data_copy/access_log_archive


In [ ]:
# Add a user-defined label to the HDFS archive DIRECTORY (not a real TTL).
# -n supplies the attribute name and -v supplies its text value.
!hdfs dfs -setfattr -n user.cn7031_module -v week2 /data_copy

In [ ]:
# Display extended attributes (-d) from the archive directory to verify the label.
!hdfs dfs -getfattr -d /data_copy


# file: /data_copy
user.cn7031_module="week2"


In [ ]:
# Count directories, files and total storage used in HDFS.
# -count displays directory count, file count and content size.
# -h converts storage sizes into human-readable units (KB, MB, GB).
!hdfs dfs -count -h /data_copy

           1            1            195.7 M /data_copy


In [ ]:
# Retrieve the filename stored in HDFS.
# %n returns the filename without its parent directory.
!hdfs dfs -stat '%n' /data_copy/access_log_archive
# Retrieve the file size in bytes.
# %b returns the logical file length.
!hdfs dfs -stat '%b' /data_copy/access_log_archive
# Retrieve the replication factor and block size.
# %r returns replication factor; %% returns block size in bytes.
!hdfs dfs -stat 'Replication: %r | Block size: %% bytes' /data_copy/access_log_archive

access_log_archive
205242368
Replication: 1 | Block size: % bytes


In [ ]:
 # Display approximately the last 1 KB of the HDFS archive.
# -tail reads the end of a file directly from HDFS.
!hdfs dfs -tail /data_copy/access_log_archive

lipper12055.iaccess.za - - [28/Jul/1995:13:32:22 -0400] "GET /images/USA-logosmall.gif HTTP/1.0" 200 234
163.205.53.14 - - [28/Jul/1995:13:32:22 -0400] "GET /shuttle/technology/images/srb_mod_compare_1-small.gif HTTP/1.0" 200 36902
maynard.isi.uconn.edu - - [28/Jul/1995:13:32:22 -0400] "GET /images/shuttle-patch-logo.gif HTTP/1.0" 200 891
163.205.53.14 - - [28/Jul/1995:13:32:22 -0400] "GET /shuttle/technology/images/srb_mod_compare_3-small.gif HTTP/1.0" 200 55666
163.205.53.14 - - [28/Jul/1995:13:32:22 -0400] "GET /shuttle/technology/images/srb_mod_compare_6-small.gif HTTP/1.0" 200 28219
163.205.53.14 - - [28/Jul/1995:13:32:23 -0400] "GET /images/KSC-logosmall.gif HTTP/1.0" 200 1204
tiger2.ocs.lsu.edu - - [28/Jul/1995:13:32:23 -0400] "GET /shuttle/missions/missions.html HTTP/1.0" 200 8677
199.0.2.27 - - [28/Jul/1995:13:32:23 -0400] "GET /images/ksclogo-medium.gif HTTP/1.0" 200 5866
tornado.umd.edu - - [28/Jul/1995:13:32:25 -0400] "GET /shuttle/missions/sts-74/sts-74-patch-small.gif HTT